#### Preparing Data

In [1]:
# import the necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import keras

In [2]:
# reading csv data
columns = [
    "capacity",
    "diameter",
    "column_height",
    "porosity",
    "ads_time",
    "C_o",
    "rho_s",
    "dp",
    "qm1",
    "kl1",
    "sorbent_cost_per_kg",
    "total_LCP",
    "emissions",
    "PFAS_eff",
    "PFAS_ads",
    ]

df1 = pd.read_csv("./training_data/ALLMAT_200k_training_data.csv", usecols=columns)
df = df1.dropna(ignore_index=True)
# df.tail()

In [3]:
# count rows before cleaning
print(f"Number of rows before cleaning: {len(df)}")

Number of rows before cleaning: 200000


In [4]:
epsilon = 1e-20
# cleaning data
df.loc[:, 'PFAS_eff'] = df['PFAS_eff'].clip(lower=0)
df = df[(df['total_LCP'] >= 0)].copy()

# Applying log transformations
df['capacity'] = np.log(df['capacity'])
df['ads_time'] = np.log(df['ads_time'])
df['C_o'] = np.log(df['C_o'])
df['dp'] = np.log(df['dp'])
df['qm1'] = np.log(df['qm1'])
df['kl1'] = np.log(df['kl1'])
df['sorbent_cost_per_kg'] = np.log(df['sorbent_cost_per_kg'])
df['total_LCP'] = np.log(df['total_LCP'])
df['emissions'] = np.log(df['emissions'])
df['PFAS_eff'] = np.log(df['PFAS_eff'] + epsilon)

# bounds
print(df.min())
print(df.max())

capacity                 2.304194
diameter                 0.955110
column_height            1.000010
porosity                 0.350001
ads_time                 3.401257
C_o                     -6.815012
rho_s                  500.004000
dp                      -8.111725
qm1                     -6.775754
kl1                      0.003334
sorbent_cost_per_kg      0.003125
total_LCP                9.997128
emissions                5.171745
PFAS_eff               -46.051702
PFAS_ads                 0.000022
dtype: float64
capacity                  8.517189
diameter                  4.000000
column_height             3.999990
porosity                  0.499998
ads_time                  7.509330
C_o                       2.302583
rho_s                  1999.980000
dp                       -5.991469
qm1                       2.302564
kl1                       9.210334
sorbent_cost_per_kg       7.600902
total_LCP                29.519682
emissions                19.495519
PFAS_eff            

In [5]:
# count rows after cleaning
print(f"Number of rows after cleaning: {len(df)}")

Number of rows after cleaning: 200000


In [6]:
# separate the data into inputs and outputs
inputs = [
    "capacity",
    "diameter",
    "column_height",
    "porosity",
    "ads_time",
    "C_o",
    "rho_s",
    "dp",
    "qm1",
    "kl1",
    "sorbent_cost_per_kg",
    ]

outputs = [
    "total_LCP",
    "emissions",
    "PFAS_eff"
]

dfin = df[inputs]
dfout = df[outputs]

In [7]:
# define a function to scale the data using min-max scaling
def min_max_scaler(data, data_min, data_max):
    return (data - data_min) /(data_max - data_min)

# define a function to reverse the scaling
def reverse_min_max_scaler(scaled_data, data_min, data_max):
    return scaled_data * (data_max - data_min) + data_min

In [8]:
# scale the data
x_data_min = dfin.min()
x_data_max = dfin.max()
y_data_min = dfout.min()
y_data_max = dfout.max()

x_data_scaled = min_max_scaler(dfin, x_data_min, x_data_max)
y_data_scaled = min_max_scaler(dfout, y_data_min, y_data_max)

x_scaled_min = x_data_scaled.min()
x_scaled_max = x_data_scaled.max()
y_scaled_min = y_data_scaled.min()
y_scaled_max = y_data_scaled.max()


#### Optimise Model with pyomo

In [9]:
import pyomo.environ as pyo
# omlt for interfacing our neural network with pyomo
from Custom_scaling import MinMaxScaling
from omlt import OffsetScaling, OmltBlock
from omlt.io.keras import keras_reader, load_keras_sequential
from omlt.neuralnet import (
    FullSpaceNNFormulation,
    FullSpaceSmoothNNFormulation,
    ReducedSpaceSmoothNNFormulation,
    ReluBigMFormulation,
    ReluComplementarityFormulation,
    ReluPartitionFormulation,
)
from omlt.neuralnet.activations import ComplementarityReLUActivation

In [10]:
# # Check solver availability
# for solver in ["glpk", "cbc", "ipopt", "baron", "cplex", "appsi_maingo", "gurobi"]:
#     print(solver, pyo.SolverFactory(solver).available())

In [11]:
# unscaling back using factors
scaler = MinMaxScaling(
        min_inputs={i: x_data_min[inputs[i]] for i in range(len(inputs))},
        max_inputs={i: x_data_max[inputs[i]] for i in range(len(inputs))},
        min_outputs={i: y_data_min[outputs[i]] for i in range(len(outputs))},
        max_outputs={i: y_data_max[outputs[i]] for i in range(len(outputs))},
    )

# saving unscaled bounds
#input
unscaled_lb = x_data_min[inputs].to_numpy()
unscaled_ub = x_data_max[inputs].to_numpy()
#output
output_lb = y_data_min[outputs].to_numpy()
output_ub = y_data_max[outputs].to_numpy()

# saving scaled bounds
# input
scaled_lb = x_scaled_min[inputs].to_numpy()
scaled_ub = x_scaled_max[inputs].to_numpy()
# output
scaled_output_lb = y_scaled_min[outputs].to_numpy()
scaled_output_ub = y_scaled_max[outputs].to_numpy()

scaled_input_bounds = {i: (scaled_lb[i], scaled_ub[i]) for i in range(len(inputs))}
unscaled_input_bounds = {i: (unscaled_lb[i], unscaled_ub[i]) for i in range(len(inputs))}

In [12]:
# Optimisation Data
obj_scaler = 1e-6

#placeholder input parameters for the optimization model
# process parameters
feed_conc_val = 0.05 # ppb
EPA_conc_val = 9.66e-9 # mol/m3
capacity_val = 1000

# materials properties
density_val = 750 # kg/m3
porosity_val = 0.4
dp_val = 0.002 # m
qm1_val = 0.5 # mol/kg
kl1_val = 10 # m3/mol
sorbent_cost_val = 2 # $/kg

# loading saved nn
nn_PFAS_model = keras.models.load_model("ALLMAT_relu_mse.keras", compile=False)
net = load_keras_sequential(
        nn_PFAS_model, scaling_object=scaler, scaled_input_bounds=scaled_input_bounds, unscaled_input_bounds=unscaled_input_bounds
    )

# Pyomo model with algebraic constraints
def solve_model(objective='cost', eps_emissions=None, feed_conc=feed_conc_val, EPA_conc=EPA_conc_val, capacity=capacity_val, density=density_val, porosity=porosity_val, dp=dp_val, qm1=qm1_val, kl1=kl1_val, sorbent_cost=sorbent_cost_val):

    max_capacity_per_bed = 120 # m3/h
    N_beds = int(np.ceil(capacity/max_capacity_per_bed))
    Q_per_column = capacity/N_beds

    optimization_model = pyo.ConcreteModel()
    
    optimization_model.diameter = pyo.Var(initialize=(unscaled_lb[1]+unscaled_ub[1])/2, bounds=(unscaled_lb[1], unscaled_ub[1]))
    optimization_model.column_height = pyo.Var(initialize=(unscaled_lb[2]+unscaled_ub[2])/2, bounds=(unscaled_lb[2], unscaled_ub[2]))
    optimization_model.porosity = pyo.Var(initialize=(unscaled_lb[3]+unscaled_ub[3])/2, bounds=(unscaled_lb[3], unscaled_ub[3]))
    optimization_model.dp = pyo.Var(initialize=(unscaled_lb[7]+unscaled_ub[7])/2, bounds=(unscaled_lb[7], unscaled_ub[7]))
    optimization_model.ads_time = pyo.Var(initialize=(unscaled_lb[4]+unscaled_ub[4])/2, bounds=(unscaled_lb[4], unscaled_ub[4]))
    
    optimization_model.total_LCP = pyo.Var(bounds=(output_lb[0], output_ub[0]))
    optimization_model.emissions = pyo.Var(bounds=(output_lb[1], output_ub[1]))
    optimization_model.PFAS_eff_conc = pyo.Var(bounds=(output_lb[2], output_ub[2]))

    optimization_model.nn = OmltBlock()
    formulated_problem = ReluBigMFormulation(net)
    optimization_model.nn.build_formulation(formulated_problem)
    
    optimization_model.nn.inputs[0].fix(pyo.log(capacity))
    optimization_model.nn.inputs[5].fix(pyo.log(feed_conc))
    optimization_model.nn.inputs[6].fix(density)
    optimization_model.nn.inputs[8].fix(pyo.log(qm1))
    optimization_model.nn.inputs[9].fix(pyo.log(kl1))
    optimization_model.nn.inputs[10].fix(pyo.log(sorbent_cost))

    optimization_model.con_in_d = pyo.Constraint(expr= optimization_model.diameter == optimization_model.nn.inputs[1])
    optimization_model.con_in_H = pyo.Constraint(expr= optimization_model.column_height == optimization_model.nn.inputs[2])
    optimization_model.con_in_p = pyo.Constraint(expr= optimization_model.porosity == optimization_model.nn.inputs[3])
    optimization_model.con_in_dp = pyo.Constraint(expr= optimization_model.dp == optimization_model.nn.inputs[7])
    optimization_model.con_in_t = pyo.Constraint(expr= optimization_model.ads_time == optimization_model.nn.inputs[4])

    optimization_model.con_out_LCP = pyo.Constraint(expr= optimization_model.total_LCP == optimization_model.nn.outputs[0])
    optimization_model.con_out_emi = pyo.Constraint(expr= optimization_model.emissions == optimization_model.nn.outputs[1])
    optimization_model.con_out_eff = pyo.Constraint(expr= optimization_model.PFAS_eff_conc == optimization_model.nn.outputs[2])

    # constraints
    optimization_model.epa_limit = pyo.Constraint(expr= optimization_model.PFAS_eff_conc <= pyo.log(EPA_conc + epsilon))
    optimization_model.min_velocity = pyo.Constraint(expr= 4*Q_per_column >= 2*(np.pi*optimization_model.diameter**2))
    optimization_model.max_velocity = pyo.Constraint(expr= 4*Q_per_column <= 15*(np.pi*optimization_model.diameter**2))

    if eps_emissions is not None:
        optimization_model.emissions_con = pyo.Constraint(expr= optimization_model.emissions <= eps_emissions)
    
    if objective == 'cost':
        optimization_model.obj = pyo.Objective(expr=(optimization_model.total_LCP), sense=pyo.minimize)
    else:
        optimization_model.obj = pyo.Objective(expr=(optimization_model.emissions), sense=pyo.minimize)

    options = {
    'MaxTime': 500,
    'epsr': 0.05,
    'epsa': 1e-4,
    'lpsol': 3,
    'cplexlibname': '"C:/Users/huberttreve/Documents/Python/Solvers/CPLEX/cplex/bin/x64_win64/cplex2212.dll"',
    'threads': 4,
    'prlevel': 1,
    'dolocal': 1
    }

    # print("# of variables: ", optimization_model.nvariables())
    # print("# of constraints: ", optimization_model.nconstraints())
    
    solver = pyo.SolverFactory('baron') 
    results = solver.solve(optimization_model, tee=False, options=options)
    
    term_cond = str(results.solver.termination_condition)
    
    # Silently check if the objective has a value
    obj_val = pyo.value(optimization_model.obj, exception=False)
    solution_found = (obj_val is not None)

    if term_cond == 'optimal':
        # print("Success: Global optimal solution found")
        status = "Optimal"
    elif term_cond == 'maxTimeLimit' and solution_found:
        # print(f"Success: Time limit hit. Feasible solution found (Objective: {obj_val:.4f}).")
        status = "Feasible (Time Limit)"
    else:
        # print(f"Solver stopped ('{term_cond}') and NO feasible solution was found.")
        status = "Infeasible"

    def get_val(var):
        # Silently check each variable, return NaN if empty
        v = pyo.value(var, exception=False)
        return v if v is not None else np.nan

    return (capacity,
            N_beds,
            get_val(optimization_model.diameter),
            get_val(optimization_model.column_height),
            get_val(optimization_model.porosity),
            np.exp(get_val(optimization_model.dp)),
            np.exp(get_val(optimization_model.ads_time)),
            feed_conc,
            density,
            qm1,
            kl1,
            sorbent_cost,
            np.exp(get_val(optimization_model.total_LCP)),
            np.exp(get_val(optimization_model.emissions)),
            np.exp(get_val(optimization_model.PFAS_eff_conc) - epsilon),
            status)

In [13]:
MW_PFAS = 414.07  # g/mol

def get_effluent_mol_m3(concentration_ppb):
    return (concentration_ppb * 1e-3) / MW_PFAS

# Optimise for GAC
qm1_value        = 0.57   # mol/kg
kl1_value        = 18    # m3/mol
density_val      = 501    # kg/m3
sorbent_cost_val = 2   # $/kg

fixed_feed_conc = 0.1   # ppb

# --- Capacity sweep ---
# Set these to the capacity bounds your OMLT surrogate was trained on (out-of-bounds runs will crash)
eps_bounds = 1e-6
capacity_min = df1['capacity'].min() * (1 + eps_bounds)     # m3/h
capacity_max = df1['capacity'].max() * (1 - eps_bounds)     # m3/h
n_capacity   = 10
capacities_m3h = np.geomspace(capacity_min, capacity_max, n_capacity)

epa_limits_ppb = [0.07, 0.02, 0.004, 0.001, 0.0001]

print(f"Fixed feed concentration: {fixed_feed_conc:.4g} ppb ")
print(f"Capacity range: {capacity_min} to {capacity_max} m3/h ({n_capacity} points)")

# Column order must match the solve_model return tuple
columns = [
    "capacity", "N_beds", "diameter", "column_height", "porosity", "dp",
    "ads_time", "feed_conc", "density", "qm1", "kl1", "sorbent_cost",
    "total_LCP", "emissions", "PFAS_eff_conc", "status"
]

def make_failed_res(Q, status_msg):
    """Placeholder tuple: NaN outputs, but keep the inputs so we know which run was skipped/failed."""
    return (
        Q, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan,
        fixed_feed_conc, density_val, qm1_value, kl1_value, sorbent_cost_val,
        np.nan, np.nan, np.nan, status_msg
    )

heatmap3_data = []
full_solutions_data = []
n_solved, n_na, n_crashed = 0, 0, 0

print("Running Experiment 3: Treatment Scale (Capacity vs. Effluent Limit)...")

for Q in capacities_m3h:
    for epa_ppb in epa_limits_ppb:

        # Only solve if the feed concentration exceeds the effluent limit
        if fixed_feed_conc <= epa_ppb:
            print(f"\nSkipping Q={Q:.4g} m3/h, EPA={epa_ppb} ppb: feed already meets limit (N/A)")
            res = make_failed_res(Q, "N/A")
            n_na += 1
        else:
            print(f"\nSolving Q={Q:.4g} m3/h, EPA={epa_ppb} ppb")
            epa_mol_m3 = get_effluent_mol_m3(epa_ppb)
            try:
                res = solve_model(
                    objective='cost',
                    eps_emissions=None,
                    capacity=Q,
                    feed_conc=fixed_feed_conc,
                    EPA_conc=epa_mol_m3,
                    qm1=qm1_value,
                    kl1=kl1_value,
                    density=density_val,
                    sorbent_cost=sorbent_cost_val
                )
                n_solved += 1
            except Exception as e:
                print(f"⚠️ Solver crashed! Likely out-of-bounds for OMLT. Moving on. Error: {e}")
                res = make_failed_res(Q, "Solver Crashed")
                n_crashed += 1

        # Map the 16-value tuple to named fields
        r = dict(zip(columns, res))

        # 1. Heatmap data
        heatmap3_data.append({
            'Capacity_m3h': Q,
            'EPA_Limit_ppb': epa_ppb,
            'Cost': r['total_LCP']
        })

        # 2. Full engineering record
        full_solutions_data.append({
            'Capacity_Constraint_m3h': Q,
            'Capacity_m3h': r['capacity'],
            'EPA_Limit_ppb': epa_ppb,
            'Feed_Conc_Constraint_ppb': fixed_feed_conc,
            'N_beds': r['N_beds'],
            'Optimal_Diameter_m': r['diameter'],
            'Optimal_Height_m': r['column_height'],
            'Bed_Porosity': r['porosity'],
            'Particle_Diameter_dp': r['dp'],
            'Ads_Time': r['ads_time'],
            'Actual_Feed_Conc_ppb': r['feed_conc'],
            'Density_kg_m3': r['density'],
            'qm1_mol_kg': r['qm1'],
            'kl1_m3_mol': r['kl1'],
            'Sorbent_Cost_$/kg': r['sorbent_cost'],
            'Levelized_Cost': r['total_LCP'],
            'Emissions_kgCO2/hr': r['emissions'],
            'PFAS_eff': r['PFAS_eff_conc'],
            'Status': r['status']
        })

# DataFrames
df_plot = pd.DataFrame(heatmap3_data)
df_heatmap_matrix = df_plot.pivot(index='Capacity_m3h', columns='EPA_Limit_ppb', values='Cost')
df_full_solutions = pd.DataFrame(full_solutions_data)

excel_filename = f'GAC_Cost_vs_Capacity_and_EffluentLimit_FixedCo_{fixed_feed_conc:.3g}ppb.xlsx'

with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    df_heatmap_matrix.to_excel(writer, sheet_name='Plotting_Matrix')
    df_full_solutions.to_excel(writer, sheet_name='Full_Solutions', index=False)

print(f"\nSweep complete: {n_solved} solved, {n_na} N/A (feed ≤ limit), {n_crashed} crashed.")
print(f"Saved Experiment 3 to {excel_filename} (Contains 'Plotting_Matrix' and 'Full_Solutions' sheets)")

Fixed feed concentration: 0.1 ppb 
Capacity range: 10.016110016099999 to 4999.975000019999 m3/h (10 points)
Running Experiment 3: Treatment Scale (Capacity vs. Effluent Limit)...

Solving Q=10.02 m3/h, EPA=0.07 ppb

Solving Q=10.02 m3/h, EPA=0.02 ppb

Solving Q=10.02 m3/h, EPA=0.004 ppb

Solving Q=10.02 m3/h, EPA=0.001 ppb

Solving Q=10.02 m3/h, EPA=0.0001 ppb

Solving Q=19.98 m3/h, EPA=0.07 ppb

Solving Q=19.98 m3/h, EPA=0.02 ppb

Solving Q=19.98 m3/h, EPA=0.004 ppb

Solving Q=19.98 m3/h, EPA=0.001 ppb

Solving Q=19.98 m3/h, EPA=0.0001 ppb

Solving Q=39.84 m3/h, EPA=0.07 ppb

Solving Q=39.84 m3/h, EPA=0.02 ppb

Solving Q=39.84 m3/h, EPA=0.004 ppb

Solving Q=39.84 m3/h, EPA=0.001 ppb

Solving Q=39.84 m3/h, EPA=0.0001 ppb

Solving Q=79.46 m3/h, EPA=0.07 ppb

Solving Q=79.46 m3/h, EPA=0.02 ppb

Solving Q=79.46 m3/h, EPA=0.004 ppb

Solving Q=79.46 m3/h, EPA=0.001 ppb

Solving Q=79.46 m3/h, EPA=0.0001 ppb

Solving Q=158.5 m3/h, EPA=0.07 ppb

Solving Q=158.5 m3/h, EPA=0.02 ppb

Solving Q=15

In [14]:
MW_PFAS = 414.07  # g/mol

def get_effluent_mol_m3(concentration_ppb):
    return (concentration_ppb * 1e-3) / MW_PFAS

# Optimise for Amberlite 400
qm1_value        = 2.92   # mol/kg
kl1_value        = 69    # m3/mol
density_val      = 670    # kg/m3
sorbent_cost_val = 10   # $/kg

fixed_feed_conc = 0.1   # ppb

# --- Capacity sweep ---
# Set these to the capacity bounds your OMLT surrogate was trained on (out-of-bounds runs will crash)
eps_bounds = 1e-6
capacity_min = df1['capacity'].min() * (1 + eps_bounds)     # m3/h
capacity_max = df1['capacity'].max() * (1 - eps_bounds)     # m3/h
n_capacity   = 10
capacities_m3h = np.geomspace(capacity_min, capacity_max, n_capacity)

epa_limits_ppb = [0.07, 0.02, 0.004, 0.001, 0.0001]

print(f"Fixed feed concentration: {fixed_feed_conc:.4g} ppb ")
print(f"Capacity range: {capacity_min} to {capacity_max} m3/h ({n_capacity} points)")

# Column order must match the solve_model return tuple
columns = [
    "capacity", "N_beds", "diameter", "column_height", "porosity", "dp",
    "ads_time", "feed_conc", "density", "qm1", "kl1", "sorbent_cost",
    "total_LCP", "emissions", "PFAS_eff_conc", "status"
]

def make_failed_res(Q, status_msg):
    """Placeholder tuple: NaN outputs, but keep the inputs so we know which run was skipped/failed."""
    return (
        Q, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan,
        fixed_feed_conc, density_val, qm1_value, kl1_value, sorbent_cost_val,
        np.nan, np.nan, np.nan, status_msg
    )

heatmap3_data = []
full_solutions_data = []
n_solved, n_na, n_crashed = 0, 0, 0

print("Running Experiment 3: Treatment Scale (Capacity vs. Effluent Limit)...")

for Q in capacities_m3h:
    for epa_ppb in epa_limits_ppb:

        # Only solve if the feed concentration exceeds the effluent limit
        if fixed_feed_conc <= epa_ppb:
            print(f"\nSkipping Q={Q:.4g} m3/h, EPA={epa_ppb} ppb: feed already meets limit (N/A)")
            res = make_failed_res(Q, "N/A")
            n_na += 1
        else:
            print(f"\nSolving Q={Q:.4g} m3/h, EPA={epa_ppb} ppb")
            epa_mol_m3 = get_effluent_mol_m3(epa_ppb)
            try:
                res = solve_model(
                    objective='cost',
                    eps_emissions=None,
                    capacity=Q,
                    feed_conc=fixed_feed_conc,
                    EPA_conc=epa_mol_m3,
                    qm1=qm1_value,
                    kl1=kl1_value,
                    density=density_val,
                    sorbent_cost=sorbent_cost_val
                )
                n_solved += 1
            except Exception as e:
                print(f"⚠️ Solver crashed! Likely out-of-bounds for OMLT. Moving on. Error: {e}")
                res = make_failed_res(Q, "Solver Crashed")
                n_crashed += 1

        # Map the 16-value tuple to named fields
        r = dict(zip(columns, res))

        # 1. Heatmap data
        heatmap3_data.append({
            'Capacity_m3h': Q,
            'EPA_Limit_ppb': epa_ppb,
            'Cost': r['total_LCP']
        })

        # 2. Full engineering record
        full_solutions_data.append({
            'Capacity_Constraint_m3h': Q,
            'Capacity_m3h': r['capacity'],
            'EPA_Limit_ppb': epa_ppb,
            'Feed_Conc_Constraint_ppb': fixed_feed_conc,
            'N_beds': r['N_beds'],
            'Optimal_Diameter_m': r['diameter'],
            'Optimal_Height_m': r['column_height'],
            'Bed_Porosity': r['porosity'],
            'Particle_Diameter_dp': r['dp'],
            'Ads_Time': r['ads_time'],
            'Actual_Feed_Conc_ppb': r['feed_conc'],
            'Density_kg_m3': r['density'],
            'qm1_mol_kg': r['qm1'],
            'kl1_m3_mol': r['kl1'],
            'Sorbent_Cost_$/kg': r['sorbent_cost'],
            'Levelized_Cost': r['total_LCP'],
            'Emissions_kgCO2/hr': r['emissions'],
            'PFAS_eff': r['PFAS_eff_conc'],
            'Status': r['status']
        })

# DataFrames
df_plot = pd.DataFrame(heatmap3_data)
df_heatmap_matrix = df_plot.pivot(index='Capacity_m3h', columns='EPA_Limit_ppb', values='Cost')
df_full_solutions = pd.DataFrame(full_solutions_data)

excel_filename = f'Amberlite_400_Cost_vs_Capacity_and_EffluentLimit_FixedCo_{fixed_feed_conc:.3g}ppb.xlsx'

with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    df_heatmap_matrix.to_excel(writer, sheet_name='Plotting_Matrix')
    df_full_solutions.to_excel(writer, sheet_name='Full_Solutions', index=False)

print(f"\nSweep complete: {n_solved} solved, {n_na} N/A (feed ≤ limit), {n_crashed} crashed.")
print(f"Saved Experiment 3 to {excel_filename} (Contains 'Plotting_Matrix' and 'Full_Solutions' sheets)")

Fixed feed concentration: 0.1 ppb 
Capacity range: 10.016110016099999 to 4999.975000019999 m3/h (10 points)
Running Experiment 3: Treatment Scale (Capacity vs. Effluent Limit)...

Solving Q=10.02 m3/h, EPA=0.07 ppb

Solving Q=10.02 m3/h, EPA=0.02 ppb

Solving Q=10.02 m3/h, EPA=0.004 ppb

Solving Q=10.02 m3/h, EPA=0.001 ppb

Solving Q=10.02 m3/h, EPA=0.0001 ppb

Solving Q=19.98 m3/h, EPA=0.07 ppb

Solving Q=19.98 m3/h, EPA=0.02 ppb

Solving Q=19.98 m3/h, EPA=0.004 ppb

Solving Q=19.98 m3/h, EPA=0.001 ppb

Solving Q=19.98 m3/h, EPA=0.0001 ppb

Solving Q=39.84 m3/h, EPA=0.07 ppb

Solving Q=39.84 m3/h, EPA=0.02 ppb

Solving Q=39.84 m3/h, EPA=0.004 ppb

Solving Q=39.84 m3/h, EPA=0.001 ppb

Solving Q=39.84 m3/h, EPA=0.0001 ppb

Solving Q=79.46 m3/h, EPA=0.07 ppb

Solving Q=79.46 m3/h, EPA=0.02 ppb

Solving Q=79.46 m3/h, EPA=0.004 ppb

Solving Q=79.46 m3/h, EPA=0.001 ppb

Solving Q=79.46 m3/h, EPA=0.0001 ppb

Solving Q=158.5 m3/h, EPA=0.07 ppb

Solving Q=158.5 m3/h, EPA=0.02 ppb

Solving Q=15

In [15]:
MW_PFAS = 414.07  # g/mol

def get_effluent_mol_m3(concentration_ppb):
    return (concentration_ppb * 1e-3) / MW_PFAS

# Optimise for Amberlite™ IRA910
qm1_value        = 3.47   # mol/kg
kl1_value        = 416    # m3/mol
density_val      = 700    # kg/m3
sorbent_cost_val = 12.5   # $/kg

fixed_feed_conc = 0.1   # ppb

# --- Capacity sweep ---
# Set these to the capacity bounds your OMLT surrogate was trained on (out-of-bounds runs will crash)
eps_bounds = 1e-6
capacity_min = df1['capacity'].min() * (1 + eps_bounds)     # m3/h
capacity_max = df1['capacity'].max() * (1 - eps_bounds)     # m3/h
n_capacity   = 10
capacities_m3h = np.geomspace(capacity_min, capacity_max, n_capacity)

epa_limits_ppb = [0.07, 0.02, 0.004, 0.001, 0.0001]

print(f"Fixed feed concentration: {fixed_feed_conc:.4g} ppb ")
print(f"Capacity range: {capacity_min} to {capacity_max} m3/h ({n_capacity} points)")

# Column order must match the solve_model return tuple
columns = [
    "capacity", "N_beds", "diameter", "column_height", "porosity", "dp",
    "ads_time", "feed_conc", "density", "qm1", "kl1", "sorbent_cost",
    "total_LCP", "emissions", "PFAS_eff_conc", "status"
]

def make_failed_res(Q, status_msg):
    """Placeholder tuple: NaN outputs, but keep the inputs so we know which run was skipped/failed."""
    return (
        Q, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan,
        fixed_feed_conc, density_val, qm1_value, kl1_value, sorbent_cost_val,
        np.nan, np.nan, np.nan, status_msg
    )

heatmap3_data = []
full_solutions_data = []
n_solved, n_na, n_crashed = 0, 0, 0

print("Running Experiment 3: Treatment Scale (Capacity vs. Effluent Limit)...")

for Q in capacities_m3h:
    for epa_ppb in epa_limits_ppb:

        # Only solve if the feed concentration exceeds the effluent limit
        if fixed_feed_conc <= epa_ppb:
            print(f"\nSkipping Q={Q:.4g} m3/h, EPA={epa_ppb} ppb: feed already meets limit (N/A)")
            res = make_failed_res(Q, "N/A")
            n_na += 1
        else:
            print(f"\nSolving Q={Q:.4g} m3/h, EPA={epa_ppb} ppb")
            epa_mol_m3 = get_effluent_mol_m3(epa_ppb)
            try:
                res = solve_model(
                    objective='cost',
                    eps_emissions=None,
                    capacity=Q,
                    feed_conc=fixed_feed_conc,
                    EPA_conc=epa_mol_m3,
                    qm1=qm1_value,
                    kl1=kl1_value,
                    density=density_val,
                    sorbent_cost=sorbent_cost_val
                )
                n_solved += 1
            except Exception as e:
                print(f"⚠️ Solver crashed! Likely out-of-bounds for OMLT. Moving on. Error: {e}")
                res = make_failed_res(Q, "Solver Crashed")
                n_crashed += 1

        # Map the 16-value tuple to named fields
        r = dict(zip(columns, res))

        # 1. Heatmap data
        heatmap3_data.append({
            'Capacity_m3h': Q,
            'EPA_Limit_ppb': epa_ppb,
            'Cost': r['total_LCP']
        })

        # 2. Full engineering record
        full_solutions_data.append({
            'Capacity_Constraint_m3h': Q,
            'Capacity_m3h': r['capacity'],
            'EPA_Limit_ppb': epa_ppb,
            'Feed_Conc_Constraint_ppb': fixed_feed_conc,
            'N_beds': r['N_beds'],
            'Optimal_Diameter_m': r['diameter'],
            'Optimal_Height_m': r['column_height'],
            'Bed_Porosity': r['porosity'],
            'Particle_Diameter_dp': r['dp'],
            'Ads_Time': r['ads_time'],
            'Actual_Feed_Conc_ppb': r['feed_conc'],
            'Density_kg_m3': r['density'],
            'qm1_mol_kg': r['qm1'],
            'kl1_m3_mol': r['kl1'],
            'Sorbent_Cost_$/kg': r['sorbent_cost'],
            'Levelized_Cost': r['total_LCP'],
            'Emissions_kgCO2/hr': r['emissions'],
            'PFAS_eff': r['PFAS_eff_conc'],
            'Status': r['status']
        })

# DataFrames
df_plot = pd.DataFrame(heatmap3_data)
df_heatmap_matrix = df_plot.pivot(index='Capacity_m3h', columns='EPA_Limit_ppb', values='Cost')
df_full_solutions = pd.DataFrame(full_solutions_data)

excel_filename = f'Amberlite_IRA910_Cost_vs_Capacity_and_EffluentLimit_FixedCo_{fixed_feed_conc:.3g}ppb.xlsx'

with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    df_heatmap_matrix.to_excel(writer, sheet_name='Plotting_Matrix')
    df_full_solutions.to_excel(writer, sheet_name='Full_Solutions', index=False)

print(f"\nSweep complete: {n_solved} solved, {n_na} N/A (feed ≤ limit), {n_crashed} crashed.")
print(f"Saved Experiment 3 to {excel_filename} (Contains 'Plotting_Matrix' and 'Full_Solutions' sheets)")

Fixed feed concentration: 0.1 ppb 
Capacity range: 10.016110016099999 to 4999.975000019999 m3/h (10 points)
Running Experiment 3: Treatment Scale (Capacity vs. Effluent Limit)...

Solving Q=10.02 m3/h, EPA=0.07 ppb

Solving Q=10.02 m3/h, EPA=0.02 ppb

Solving Q=10.02 m3/h, EPA=0.004 ppb

Solving Q=10.02 m3/h, EPA=0.001 ppb

Solving Q=10.02 m3/h, EPA=0.0001 ppb

Solving Q=19.98 m3/h, EPA=0.07 ppb

Solving Q=19.98 m3/h, EPA=0.02 ppb

Solving Q=19.98 m3/h, EPA=0.004 ppb

Solving Q=19.98 m3/h, EPA=0.001 ppb

Solving Q=19.98 m3/h, EPA=0.0001 ppb

Solving Q=39.84 m3/h, EPA=0.07 ppb

Solving Q=39.84 m3/h, EPA=0.02 ppb

Solving Q=39.84 m3/h, EPA=0.004 ppb

Solving Q=39.84 m3/h, EPA=0.001 ppb

Solving Q=39.84 m3/h, EPA=0.0001 ppb

Solving Q=79.46 m3/h, EPA=0.07 ppb

Solving Q=79.46 m3/h, EPA=0.02 ppb

Solving Q=79.46 m3/h, EPA=0.004 ppb

Solving Q=79.46 m3/h, EPA=0.001 ppb

Solving Q=79.46 m3/h, EPA=0.0001 ppb

Solving Q=158.5 m3/h, EPA=0.07 ppb

Solving Q=158.5 m3/h, EPA=0.02 ppb

Solving Q=15